In [ ]:
pip install faker pandas

In [ ]:
import pandas as pd
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta

# Inicializa o Faker para Português do Brasil
fake = Faker('pt_BR')

# --- CONFIGURAÇÃO ---
# Quantidade de registros a gerar
QTD_CLIENTES = 500
QTD_CARROS = 600  # Alguns clientes têm mais de um carro
QTD_LAVAGENS_FATO = 10000 # Número de registros na tabela fato
DATA_INICIO_SIMULACAO = datetime(2023, 1, 1)
DATA_FIM_SIMULACAO = datetime(2024, 5, 31) # Simula cerca de 1 ano e meio

print(f"Iniciando geração de dados fictícios para Lava Jato...")
print(f"Período: {DATA_INICIO_SIMULACAO.date()} até {DATA_FIM_SIMULACAO.date()}")

# ---------------------------------------------------------
# 1. DIMENSÃO: SERVIÇOS (Dados estáticos e realistas)
# ---------------------------------------------------------
print("Gerando Dimensão Serviços...")
servicos_data = [
    {"Tipo_Servico": "Lavagem Simples", "Categoria_Servico": "Padrão", "Preco_Base": 50.00},
    {"Tipo_Servico": "Lavagem Completa", "Categoria_Servico": "Padrão", "Preco_Base": 80.00},
    {"Tipo_Servico": "Lavagem Motor", "Categoria_Servico": "Especial", "Preco_Base": 100.00},
    {"Tipo_Servico": "Polimento Comercial", "Categoria_Servico": "Estética", "Preco_Base": 250.00},
    {"Tipo_Servico": "Polimento Técnico", "Categoria_Servico": "Estética", "Preco_Base": 500.00},
    {"Tipo_Servico": "Higienização Interna", "Categoria_Servico": "Estética", "Preco_Base": 350.00},
    {"Tipo_Servico": "Vitrificação Pintura", "Categoria_Servico": "Premium", "Preco_Base": 1200.00},
]
df_servicos = pd.DataFrame(servicos_data)
df_servicos['ID_Servico'] = [i + 1 for i in range(len(servicos_data))] # IDs sequenciais


# ---------------------------------------------------------
# 2. DIMENSÃO: CARROS (Mistura de Faker e dados estáticos)
# ---------------------------------------------------------
print("Gerando Dimensão Carros...")
# Dados de apoio para carros comuns no Brasil
marcas_modelos = {
    "Fiat": [("Argo", "Pequeno"), ("Mobi", "Pequeno"), ("Toro", "Grande"), ("Strada", "Médio")],
    "Volkswagen": [("Gol", "Pequeno"), ("Polo", "Pequeno"), ("T-Cross", "Médio"), ("Amarok", "Grande")],
    "Chevrolet": [("Onix", "Pequeno"), ("Tracker", "Médio"), ("S10", "Grande"), ("Spin", "Médio")],
    "Hyundai": [("HB20", "Pequeno"), ("Creta", "Médio")],
    "Toyota": [("Corolla", "Médio"), ("Hilux", "Grande"), ("Yaris", "Pequeno")]
}
marcas = list(marcas_modelos.keys())

carros_list = []
for _ in range(QTD_CARROS):
    marca = random.choice(marcas)
    modelo_info = random.choice(marcas_modelos[marca])

    carros_list.append({
        "ID_Carro": str(uuid.uuid4())[:8], # ID curto aleatório
        "Marca": marca,
        "Modelo": modelo_info[0],
        "Categoria": modelo_info[1],
        "Placa": fake.bothify(text='???-####').upper(), # Gera placa antiga (AAA-1234)
        "Ano": random.randint(2010, 2024)
    })
df_carros = pd.DataFrame(carros_list)


# ---------------------------------------------------------
# 3. DIMENSÃO: CLIENTES (Para enriquecer a análise)
# ---------------------------------------------------------
print("Gerando Dimensão Clientes...")
clientes_list = []
for _ in range(QTD_CLIENTES):
    clientes_list.append({
        "ID_Cliente": str(uuid.uuid4())[:8],
        "Nome_Cliente": fake.name(),
        "Cidade": fake.city(),
        "Estado": fake.state_abbr(),
        "Tipo_Cliente": random.choice(["Pessoa Física", "Pessoa Física", "Pessoa Jurídica"]) # Mais PF que PJ
    })
df_clientes = pd.DataFrame(clientes_list)


# ---------------------------------------------------------
# 4. TABELA FATO: LAVAGENS
# ---------------------------------------------------------
print(f"Gerando Tabela Fato Lavagens ({QTD_LAVAGENS_FATO} registros)...")
fato_list = []

# Preparar IDs para escolha aleatória eficiente
ids_carros = df_carros['ID_Carro'].tolist()
ids_servicos = df_servicos['ID_Servico'].tolist()
precos_servicos = df_servicos.set_index('ID_Servico')['Preco_Base'].to_dict()

# Mapeamento de Categoria do Carro para Multiplicador de Preço (carro grande paga mais)
multiplicador_categoria = {
    "Pequeno": 1.0,
    "Médio": 1.15,
    "Grande": 1.30
}
carro_categoria_map = df_carros.set_index('ID_Carro')['Categoria'].to_dict()


current_date = DATA_INICIO_SIMULACAO
dias_totais = (DATA_FIM_SIMULACAO - DATA_INICIO_SIMULACAO).days

# Tenta distribuir as lavagens ao longo dos dias, com sazonalidade
for i in range(QTD_LAVAGENS_FATO):
    # Sazonalidade simples: mais lavagens no Fim de semana (Sexta, Sábado)
    # E menos nos meses de chuva (ex: Jan, Fev no Sudeste)

    data_lavagem = DATA_INICIO_SIMULACAO + timedelta(days=random.randint(0, dias_totais))
    dia_semana = data_lavagem.weekday() # 0=Segunda, 6=Domingo
    mes = data_lavagem.month

    # Peso para fim de semana
    peso_dia = 1.5 if dia_semana >= 4 else 1.0
    # Peso para meses chuvosos (Jan, Fev, Dez) - reduzindo movimento
    peso_mes = 0.7 if mes in [1, 2, 12] else 1.0

    # Aplica os pesos para decidir se a lavagem "ocorre" nesta data aleatória
    # (Pula algumas iterações aleatoriamente baseada nos pesos)
    if random.random() > (0.5 * peso_dia * peso_mes):
        # Gera IDs
        id_carro = random.choice(ids_carros)
        id_servico = random.choice(ids_servicos)

        # Cálculo do valor final (Preço Base do Serviço x Multiplicador da Categoria do Carro)
        categoria_carro = carro_categoria_map[id_carro]
        preco_base = precos_servicos[id_servico]
        valor_final = round(preco_base * multiplicador_categoria[categoria_carro], 2)

        fato_list.append({
            "ID_Lavagem": i + 1,
            "ID_Carro": id_carro,
            "ID_Servico": id_servico,
            "Data_Lavagem": data_lavagem.date(),
            "Valor_Pago": valor_final,
            "Quantidade": 1 # Granularidade: 1 linha = 1 serviço
        })

df_fato = pd.DataFrame(fato_list)

# Corrige a quantidade real gerada caso a sazonalidade tenha removido muitos registros
print(f"Total de registros fatais realmente gerados: {len(df_fato)}")

# ---------------------------------------------------------
# 5. SALVAR ARQUIVOS CSV
# ---------------------------------------------------------
print("Salvando arquivos CSV...")
df_servicos.to_csv('dim_servicos.csv', index=False, sep=';', encoding='utf-8-sig')
df_carros.to_csv('dim_carros.csv', index=False, sep=';', encoding='utf-8-sig')
df_clientes.to_csv('dim_clientes.csv', index=False, sep=';', encoding='utf-8-sig')
df_fato.to_csv('fato_lavagens.csv', index=False, sep=';', encoding='utf-8-sig')

print("\nConcluído! Os seguintes arquivos foram gerados:")
print("- dim_servicos.csv")
print("- dim_carros.csv")
print("- dim_clientes.csv")
print("- fato_lavagens.csv")
print("\nVocê pode agora usar esses arquivos no seu pipeline ETL para povoar seu Data Mart.")







In [ ]:
pip install  sqlalchemy psycopg2-binary

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///lavajato_datamart.db')

print("Conectado ao banco de dados!")

print("Lendo os arquivos CSV...")
df_servicos = pd.read_csv('dim_servicos.csv', sep=';')
df_carros = pd.read_csv('dim_carros.csv', sep=';')
df_clientes = pd.read_csv('dim_clientes.csv', sep=';')
df_fato = pd.read_csv('fato_lavagens.csv', sep=';')

print("Criando a Dimensão Tempo...")
datas_unicas = pd.to_datetime(df_fato['Data_Lavagem']).dt.date.unique()
df_tempo = pd.DataFrame({'Data': datas_unicas})
df_tempo['Data'] = pd.to_datetime(df_tempo['Data'])
df_tempo['ID_Tempo'] = df_tempo['Data'].dt.strftime('%Y%m%d').astype(int)
df_tempo['Dia'] = df_tempo['Data'].dt.day
df_tempo['Mes'] = df_tempo['Data'].dt.month
df_tempo['Ano'] = df_tempo['Data'].dt.year
df_tempo['Dia_da_Semana'] = df_tempo['Data'].dt.weekday
df_tempo['Fim_de_Semana'] = df_tempo['Dia_da_Semana'].apply(lambda x: 1 if x >= 5 else 0)

df_fato['Data_Lavagem'] = pd.to_datetime(df_fato['Data_Lavagem'])
df_fato['ID_Tempo'] = df_fato['Data_Lavagem'].dt.strftime('%Y%m%d').astype(int)
df_fato = df_fato.drop(columns=['Data_Lavagem'])

print("Carregando dados para o banco SQL...")
df_servicos.to_sql('dim_servico', engine, if_exists='replace', index=False)
df_carros.to_sql('dim_carro', engine, if_exists='replace', index=False)
df_clientes.to_sql('dim_cliente', engine, if_exists='replace', index=False)
df_tempo.to_sql('dim_tempo', engine, if_exists='replace', index=False)

df_fato.to_sql('fato_lavagens', engine, if_exists='replace', index=False)

print("ETL concluído com sucesso! Banco de dados populado.")
